# Environment check — DAT09-06 Representation Learning

**Run this before session 1.** It takes about a minute.

The point is to discover a broken install *at home*, where you have time to fix
it, rather than in the first ten minutes of a lab. Run every cell top to bottom.
The last cell prints a verdict.

If something fails, the cell tells you what to do about it.

---

**Setup, if you have not done it yet:**

```bash
python -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install --upgrade pip
pip install -r requirements.txt
python data/prepare_data.py --all
```

On Linux, add `--extra-index-url https://download.pytorch.org/whl/cpu` to the
`pip install` so you do not download 2.5 GB of CUDA libraries you will never use.

## 1. Python version

The course needs Python 3.10 or newer.

In [ ]:
import platform
import sys

checks = {}

version = sys.version_info
ok = (version.major, version.minor) >= (3, 10)
checks["python"] = ok

print(f"Python     {platform.python_version()}")
print(f"Executable {sys.executable}")
print(f"Platform   {platform.platform()}")
print()
print("OK" if ok else "TOO OLD — this course needs Python >= 3.10")

## 2. Dependencies

Every package the six sessions import.

In [ ]:
import importlib

REQUIRED = [
    ("torch", "deep learning, all sessions"),
    ("torchvision", "Fashion-MNIST and image transforms, sessions 1-3"),
    ("numpy", "everywhere"),
    ("scipy", "linear algebra, session 5"),
    ("sklearn", "probes and clustering metrics, all sessions"),
    ("pandas", "metadata and comparison tables, sessions 4-6"),
    ("matplotlib", "every figure"),
    ("sentence_transformers", "frozen text encoder, sessions 4-6"),
    ("faiss", "vector index, session 6"),
    ("umap", "latent visualisation, sessions 1-3"),
    ("PIL", "product thumbnails, sessions 4-6"),
    ("tqdm", "progress bars"),
]

missing = []
print(f"{'package':<24s} {'version':<12s} used for")
print("-" * 78)

for name, purpose in REQUIRED:
    try:
        module = importlib.import_module(name)
        ver = getattr(module, "__version__", "?")
        print(f"{name:<24s} {ver:<12s} {purpose}")
    except ImportError:
        missing.append(name)
        print(f"{name:<24s} {'MISSING':<12s} {purpose}")

checks["imports"] = not missing
print()
if missing:
    print(f"Missing: {', '.join(missing)}")
    print("Fix with:  pip install -r requirements.txt")
else:
    print("OK — everything imports")

## 3. PyTorch, on the CPU

This course is CPU-only by design, so we are not checking for a GPU — we are
checking that PyTorch works and that tensor arithmetic gives the right answers.

We also pin the thread count. Left alone, PyTorch grabs every core, which on a
laptop with efficiency cores is often *slower* and always makes timings
unpredictable.

In [ ]:
import os

import torch

torch.set_num_threads(min(4, os.cpu_count() or 1))

print(f"torch          {torch.__version__}")
print(f"CUDA available {torch.cuda.is_available()}   (expected: False — we do not need it)")
print(f"CPU cores      {os.cpu_count()}")
print(f"torch threads  {torch.get_num_threads()}")
print()

# Does autograd actually work?
x = torch.randn(64, 32)
layer = torch.nn.Linear(32, 16)
loss = layer(x).pow(2).mean()
loss.backward()

grad_ok = layer.weight.grad is not None and torch.isfinite(layer.weight.grad).all()
checks["torch"] = bool(grad_ok)

print(f"forward pass   loss = {loss.item():.4f}")
print(f"backward pass  gradients finite: {bool(grad_ok)}")
print()
print("OK" if grad_ok else "BROKEN — reinstall torch")

## 4. A real training loop

A tiny autoencoder on random data, for a few hundred steps. This is the closest
thing to what you will actually do in class, and it gives you a rough sense of
how fast your machine is.

The loss should drop clearly. If it does not, something is wrong with your
install, not with your luck.

In [ ]:
import time

import torch.nn as nn

torch.manual_seed(42)

# Low-rank data, so a bottleneck of 8 can genuinely reconstruct it.
latent_true = torch.randn(2048, 8)
basis = torch.randn(8, 128)
data = latent_true @ basis + 0.1 * torch.randn(2048, 128)

model = nn.Sequential(
    nn.Linear(128, 32), nn.ReLU(), nn.Linear(32, 8),   # encoder
    nn.Linear(8, 32), nn.ReLU(), nn.Linear(32, 128),   # decoder
)
optimiser = torch.optim.Adam(model.parameters(), lr=1e-2)

start = time.perf_counter()
losses = []

for step in range(300):
    batch = data[torch.randint(0, len(data), (128,))]
    loss = nn.functional.mse_loss(model(batch), batch)

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()

    losses.append(loss.item())
    if step % 60 == 0:
        print(f"step {step:>3d}   loss {loss.item():8.3f}")

elapsed = time.perf_counter() - start
improved = losses[-1] < 0.5 * losses[0]
checks["training"] = bool(improved)

print()
print(f"300 steps in {elapsed:.1f} s  ({1000 * elapsed / 300:.1f} ms/step)")
print(f"loss {losses[0]:.3f} -> {losses[-1]:.3f}")
print()
print("OK — the loss went down" if improved else "SUSPICIOUS — the loss barely moved")

## 5. Course utilities

Every notebook imports from `common/`, which lives at the repository root. If
this cell fails, you are almost certainly running the notebook from the wrong
directory — open it from inside the cloned repository.

In [ ]:
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

try:
    import numpy as np

    from common import REPO_ROOT, evaluate_representation, set_seed, use_course_style

    set_seed()
    use_course_style()

    # Two obviously different representations of the same labels: the first
    # carries the class, the second is noise. The scores must reflect that.
    rng = np.random.default_rng(0)
    y = rng.integers(0, 4, size=800)
    informative = rng.normal(y[:, None] * 3.0, 1.0, size=(800, 6))
    uninformative = rng.normal(0.0, 1.0, size=(800, 6))

    good = evaluate_representation(informative, y, name="informative")
    bad = evaluate_representation(uninformative, y, name="noise")

    sane = good.probe_accuracy > 0.9 and bad.probe_accuracy < 0.5
    checks["common"] = bool(sane)

    print()
    print(f"repository root: {REPO_ROOT}")
    print("OK" if sane else "UNEXPECTED — the probe scores make no sense")

except ImportError as exc:
    checks["common"] = False
    print(f"Could not import common/: {exc}")
    print()
    print("Open this notebook from the root of the cloned repository —")
    print(f"the folder containing README.md. You are currently in:\n  {Path.cwd()}")

## 6. Figures

A plot should appear below, in the course palette.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(losses, linewidth=1.6)
ax.set_xlabel("step")
ax.set_ylabel("reconstruction loss")
ax.set_title("Your environment, training a very small autoencoder")
ax.set_yscale("log")
plt.show()

checks["plotting"] = True
print("OK — if you can see a decreasing curve above")

## 7. Datasets

Sessions 1–3 use Fashion-MNIST; sessions 4–6 use a product catalog subset with
precomputed image embeddings.

This is the one check that is allowed to fail without being your fault: if the
data is not built yet, the cell tells you the single command that builds it.

In [ ]:
# Fashion-MNIST is the one thing you build yourself: one command, 30 MB.
# Later sessions use larger data that is added to the repository ready-made.

EXPECTED = [
    ("fmnist_train.npz", "Fashion-MNIST train", "sessions 1-3"),
    ("fmnist_test.npz", "Fashion-MNIST test", "sessions 1-3"),
]

processed = Path.cwd() / "data" / "processed"
found = 0

print(f"  {'file':<26s} {'size':>10s}   needed for")
print("-" * 66)

for filename, description, when in EXPECTED:
    path = processed / filename
    if path.exists():
        size = f"{path.stat().st_size / 1e6:.1f} MB"
        found += 1
    else:
        size = "missing"
    print(f"  {filename:<26s} {size:>10s}   {when} - {description}")

checks["data"] = found == len(EXPECTED)

print()
if checks["data"]:
    print("OK - the datasets are in place")
else:
    print(f"{len(EXPECTED) - found} of {len(EXPECTED)} missing. Build them with:")
    print("    python data/prepare_data.py --all")


## Verdict

In [ ]:
LABELS = {
    "python": "Python version",
    "imports": "Dependencies",
    "torch": "PyTorch and autograd",
    "training": "A real training loop",
    "common": "Course utilities",
    "plotting": "Figures",
    "data": "Datasets",
}

# Missing datasets are not an environment problem — you just have not run the
# build command yet — so that row gets its own wording rather than a scary FAIL.
print("=" * 46)
for key, label in LABELS.items():
    status = checks.get(key)
    if status is True:
        mark = "pass"
    elif status is None:
        mark = "not run"
    elif key == "data":
        mark = "not built yet"
    else:
        mark = "FAIL"
    print(f"  {label:<32s} {mark}")
print("=" * 46)
print()

blocking = [k for k in LABELS if checks.get(k) is not True and k != "data"]

if not blocking and checks.get("data"):
    print("You are ready. See you in session 2, on Monday 9 November.")
elif not blocking:
    print("Your environment is fine. One thing left:")
    print("    python data/prepare_data.py --all")
else:
    print("Fix these before session 2:")
    for key in blocking:
        print(f"  - {LABELS[key]}")
    print()
    print("If you are stuck, email the output of this notebook before Sunday evening.")